In [1]:
print("HELLO")

HELLO


In [4]:
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt
from langchain_groq import ChatGroq
from langchain_classic.prompts import PromptTemplate
import os
from dotenv import load_dotenv
load_dotenv()

key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=key)



In [ ]:
from typing import List, Annotated
from pydantic import BaseModel
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, AIMessage, SystemMessage, HumanMessage
from pydantic import Field


class UserDetails(BaseModel):
    name: str
    lat: float
    long: float
    age: int
    language: str
    profession: str


class OnboardingState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    profile: UserDetails
    language: str


class InterviewState(BaseModel):
    passes: int
    total_score: int
    inst_score: int
    question: str
    answer: str
    messages: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    summary: str


In [25]:
from typing import Annotated
from pydantic import BaseModel, Field
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate

class UserDetails(BaseModel):
    name: str
    lat: float
    long: float
    age: int
    language: str
    profession: str



class InterviewState(BaseModel):
    passes: int = 0
    total_score: int = 0
    inst_score: int = 0
    user: UserDetails
    question: str = ""
    answer: str = ""

    messages: Annotated[list[BaseMessage], add_messages] = Field(
        default_factory=list
    )

    summary: str = ""


class Evaluator(BaseModel):
    score: float=Field(description="Float score from 1-10 based on how good the user's response to the question is.")

def ask_question(state: InterviewState):

    prompt = PromptTemplate.from_template("""
You are a conversational onboarding agent for a local employment-matching system.

Your task is to generate **the next best question** to ask a local worker based on the conversation so far.

The worker's basic information such as **name, age, profession, location, language, and other demographic details may already be available**. Do not ask for information that has already been provided.

You will receive the **previous conversation/messages as reference**. Carefully analyze them before generating the next question.

### Your Goal

Gradually understand the worker's **actual work experience, practical skills, responsibilities, methods, tools, strengths, and capabilities** so the information can later be used to match them with suitable employment opportunities.

### Rules

1. Ask **only ONE question at a time**.
2. The question must be **directly relevant to the previous answers**.
3. Prefer follow-up questions that dig deeper into something the worker has already mentioned.
4. Do not repeat questions or ask for information already present in the conversation.
5. If the worker mentions a specific job, task, tool, skill, or experience, use it to formulate a more specific follow-up.
6. Focus on **what they actually did and how they did it**, rather than generic questions about their skills.
7. Keep questions **short, simple, and conversational**, suitable for a local worker.
8. Avoid technical, corporate, or complicated language.
9. Prioritize information useful for **job matching**, such as:

   * Previous work performed
   * Specific responsibilities
   * Tasks they can independently handle
   * Tools or equipment they have used
   * Techniques or methods they know
   * Problems they have solved
   * Experience with different types of work
   * Strengths demonstrated through real work
10. Do not ask multiple questions in one message.
11. If the previous conversation already contains sufficient information about a topic, move to the next most useful missing area.
12. Never invent facts about the worker.
13. The conversation should feel like a **natural interview**, not a questionnaire.

### Input

You will receive the previous conversation/messages as context.
message history:
{messages}

user's name:
{name}

User's Profession:
{profession}

User's age:
{age}
### Output

Return **only the next question** to ask the worker.

Do not provide explanations, analysis, question numbers, or multiple alternatives.

""")
    chain = prompt | llm | StrOutputParser()
    question = chain.invoke({"messages":state.messages, "profession":state.user.profession, "age":state.user.age, "name":state.user.name})

    return {
        "question": question,
        "messages": [
            AIMessage(content=question)
        ]
    }


def get_answer(state: InterviewState):

    print("\nQuestion:", state.question)

    answer = input("Answer: ")

    return {
        "answer": answer,
        "messages": [
            HumanMessage(content=answer)
        ]
    }


def evaluate_answer(state: InterviewState):

    print("Evaluating:", state.answer)

    answer = state.answer
    question = state.question
    evaluator_prompt = PromptTemplate.from_template("""
You are an onboarding evaluator agent. Your task is to score the user's answer based on relevance and accuracy.
Question: {question}
\n
Answer: {answer}
""")

    structured_llm = llm.with_structured_output(Evaluator)
    chain = evaluator_prompt | structured_llm 
    score = chain.invoke({
        "answer":answer,
        "question":question
    })

    passes = state.passes + 1

    return {
        "inst_score": score.score,
        "total_score": state.total_score + score.score,
        "passes": passes
    }


def check_passes(state: InterviewState):

    if state.passes >= 3:
        return "end"

    return "continue"


builder = StateGraph(InterviewState)

builder.add_node("ask_question", ask_question)
builder.add_node("get_answer", get_answer)
builder.add_node("evaluate_answer", evaluate_answer)

builder.add_edge(START, "ask_question")
builder.add_edge("ask_question", "get_answer")
builder.add_edge("get_answer", "evaluate_answer")

builder.add_conditional_edges(
    "evaluate_answer",
    check_passes,
    {
        "continue": "ask_question",
        "end": END
    }
)

graph = builder.compile()

In [26]:
user = {
    "name":"Tanishq",
    "lat":24,
    "long":18,
    "age":21,
    "language":"hi",
    "profession":"Gaming Truck driver"
}



m1 = [SystemMessage(content="This is the start of onboarding system.")]
graph.invoke({
    "passes":0,
    "total_score":0,
    "inst_score":0,
    "user":user,
    "question":"",
    "answer":"",
    "messages":m1,
    "summary":""
    
})


Question: What are the main tasks you handle when you’re driving and setting up the gaming truck?
Evaluating: logistics

Question: Can you walk me through how you load, transport, and set up the gaming equipment for each event?
Evaluating: get and send

Question: When you arrive at a venue, what exact steps do you follow to set up the gaming stations and get everything running?
Evaluating: thats when its done 


{'passes': 3,
 'total_score': 4.0,
 'inst_score': 1.0,
 'user': {'name': 'Tanishq',
  'lat': 24,
  'long': 18,
  'age': 21,
  'language': 'hi',
  'profession': 'Gaming Truck driver'},
 'question': 'When you arrive at a venue, what exact steps do you follow to set up the gaming stations and get everything running?',
 'answer': 'thats when its done ',
 'messages': [SystemMessage(content='This is the start of onboarding system.', additional_kwargs={}, response_metadata={}, id='d6c0104a-4b21-4d38-a7af-958e5ecc7110'),
  AIMessage(content='What are the main tasks you handle when you’re driving and setting up the gaming truck?', additional_kwargs={}, response_metadata={}, id='2bc4e431-733c-4de1-8996-b6f2475dc270', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='logistics', additional_kwargs={}, response_metadata={}, id='c7b457d5-26dc-402e-9a64-3f5e8ee4e585'),
  AIMessage(content='Can you walk me through how you load, transport, and set up the gaming equipment for each event?', 

In [27]:
graph = StateGraph(InterviewState)



In [42]:


class ProfessionConfidence(BaseModel):
    confidence: float = Field(description="How well the LLM(you) understand the given profession")



def profession_confidence(profession):
    confidence_prompt = PromptTemplate.from_template("""
You are part of a user onboarding system that evaluates how well a worker knows their own profession.

Before you can generate good questions, you must first judge yourself: do you know enough about this profession to ask specific, relevant experience and knowledge questions about it?

Profession: {profession}

### Instructions

1. Think about whether this profession involves tasks, tools, and terminology you're familiar with.
2. If it's a common, well-documented profession (e.g. electrician, driver, tailor, cook), you likely have strong knowledge.
3. If it's vague, highly niche, informal, or region-specific in a way you can't reliably reason about, your knowledge is weak.
4. Do not guess or pretend to know more than you do — an honest low score is better than a wrong high score, since it will change how the user is questioned.

### Output

Return a confidence score from 1-10 for how well you know this profession well enough to ask specific, meaningful questions about it.
""")
     
    structured_llm = llm.with_structured_output(ProfessionConfidence)
    chain = confidence_prompt | structured_llm
    score = chain.invoke({"profession":profession})
    return{
        "confidence":score.confidence
    }



    

In [47]:
profession_confidence("rally worker")

{'confidence': 5.0}

In [ ]:
from dataclasses import dataclass
from heapq import heappush, heappop
from itertools import count


LOCATIONS = ("D", "C", "W")


@dataclass(frozen=True)
class State:
    monkey: str
    b1: str
    b2: str
    elevation: str       
    stacked: bool
    grasped: bool


INITIAL = State(
    monkey="D",
    b1="W",
    b2="D",
    elevation="F",
    stacked=False,
    grasped=False
)


def is_goal(state):
    return (
        state.monkey == "C"
        and state.b1 == "C"
        and state.b2 == "C"
        and state.elevation == "B1"
        and state.stacked
        and state.grasped
    )


def walk(state, destination):
    if state.elevation != "F":
        return None

    if destination == state.monkey:
        return None

    return State(
        monkey=destination,
        b1=state.b1,
        b2=state.b2,
        elevation="F",
        stacked=state.stacked,
        grasped=state.grasped
    )


def push_b1(state, destination):
    if state.elevation != "F":
        return None

    if state.stacked:
        return None

    if state.monkey != state.b1:
        return None

    if destination == state.monkey:
        return None

    return State(
        monkey=destination,
        b1=destination,
        b2=state.b2,
        elevation="F",
        stacked=False,
        grasped=False
    )


def push_b2(state, destination):
    if state.elevation != "F":
        return None

    if state.stacked:
        return None

    if state.monkey != state.b2:
        return None

    if destination == state.monkey:
        return None

    return State(
        monkey=destination,
        b1=state.b1,
        b2=destination,
        elevation="F",
        stacked=False,
        grasped=False
    )


def stack_boxes(state):
    if state.elevation != "F":
        return None

    if state.stacked:
        return None

    if state.monkey != "C":
        return None

    if state.b1 != "C" or state.b2 != "C":
        return None

    return State(
        monkey="C",
        b1="C",
        b2="C",
        elevation="F",
        stacked=True,
        grasped=False
    )


def climb_b2(state):
    if state.elevation != "F":
        return None

    if not state.stacked:
        return None

    if state.monkey != "C":
        return None

    return State(
        monkey="C",
        b1="C",
        b2="C",
        elevation="B2",
        stacked=True,
        grasped=False
    )


def climb_b1(state):
    if state.elevation != "B2":
        return None

    if not state.stacked:
        return None

    if state.monkey != "C":
        return None

    return State(
        monkey="C",
        b1="C",
        b2="C",
        elevation="B1",
        stacked=True,
        grasped=False
    )


def grasp(state):
    if state.elevation != "B1":
        return None

    if not state.stacked:
        return None

    if state.monkey != "C":
        return None

    if state.b1 != "C" or state.b2 != "C":
        return None

    if state.grasped:
        return None

    return State(
        monkey="C",
        b1="C",
        b2="C",
        elevation="B1",
        stacked=True,
        grasped=True
    )


ACTION_COSTS = {
    "Walk": 1,
    "Push B1": 2,
    "Push B2": 2,
    "Stack": 3,
    "Climb B2": 1,
    "Climb B1": 1,
    "Grasp": 1
}


def successors(state):
    result = []

    # Walk
    if state.elevation == "F":
        for location in LOCATIONS:
            next_state = walk(state, location)

            if next_state is not None:
                result.append(
                    ("Walk " + state.monkey + " -> " + location,
                     ACTION_COSTS["Walk"],
                     next_state)
                )

    # Push B1
    if state.elevation == "F" and not state.stacked:
        for location in LOCATIONS:
            next_state = push_b1(state, location)

            if next_state is not None:
                result.append(
                    ("Push B1 " + state.b1 + " -> " + location,
                     ACTION_COSTS["Push B1"],
                     next_state)
                )

    # Push B2
    if state.elevation == "F" and not state.stacked:
        for location in LOCATIONS:
            next_state = push_b2(state, location)

            if next_state is not None:
                result.append(
                    ("Push B2 " + state.b2 + " -> " + location,
                     ACTION_COSTS["Push B2"],
                     next_state)
                )

    # Stack
    next_state = stack_boxes(state)

    if next_state is not None:
        result.append(
            ("Stack B1 on B2",
             ACTION_COSTS["Stack"],
             next_state)
        )

    # Climb B2
    next_state = climb_b2(state)

    if next_state is not None:
        result.append(
            ("Climb onto B2",
             ACTION_COSTS["Climb B2"],
             next_state)
        )

    # Climb B1
    next_state = climb_b1(state)

    if next_state is not None:
        result.append(
            ("Climb onto B1",
             ACTION_COSTS["Climb B1"],
             next_state)
        )

    # Grasp
    next_state = grasp(state)

    if next_state is not None:
        result.append(
            ("Grasp banana",
             ACTION_COSTS["Grasp"],
             next_state)
        )

    return result


def movement_lower_bound(state):
    """
    Exact minimum cost in a relaxed problem containing only
    Walk and Push operations, with the objective of getting
    both boxes to Centre.
    """

    start = (state.monkey, state.b1, state.b2)

    queue = []
    counter = count()

    heappush(queue, (0, next(counter), start))

    distances = {start: 0}

    while queue:
        cost, _, current = heappop(queue)

        if cost != distances[current]:
            continue

        monkey, b1, b2 = current

        if b1 == "C" and b2 == "C":
            return cost

        # Relaxed Walk
        for destination in LOCATIONS:
            if destination == monkey:
                continue

            next_position = (destination, b1, b2)
            new_cost = cost + 1

            if new_cost < distances.get(next_position, float("inf")):
                distances[next_position] = new_cost
                heappush(
                    queue,
                    (new_cost, next(counter), next_position)
                )

        # Relaxed Push B1
        if monkey == b1:
            for destination in LOCATIONS:
                if destination == monkey:
                    continue

                next_position = (destination, destination, b2)
                new_cost = cost + 2

                if new_cost < distances.get(next_position, float("inf")):
                    distances[next_position] = new_cost
                    heappush(
                        queue,
                        (new_cost, next(counter), next_position)
                    )

        # Relaxed Push B2
        if monkey == b2:
            for destination in LOCATIONS:
                if destination == monkey:
                    continue

                next_position = (destination, b1, destination)
                new_cost = cost + 2

                if new_cost < distances.get(next_position, float("inf")):
                    distances[next_position] = new_cost
                    heappush(
                        queue,
                        (new_cost, next(counter), next_position)
                    )

    return float("inf")


def heuristic(state):
    if state.grasped:
        return 0

    if state.elevation == "B1":
        return 1

    if state.stacked:
        return 2

    movement_cost = movement_lower_bound(state)

    # Conservative phase lower bound.
    # Stack + climbing + grasp requires at least 5.
    phase_cost = 5

    return movement_cost + phase_cost


def state_string(state):
    stack = "S" if state.stacked else "-"
    grasped = "G" if state.grasped else "-"

    return (
        f"({state.monkey}, {state.b1}, {state.b2}, "
        f"{state.elevation}, {stack}, {grasped})"
    )


def a_star():
    counter = count()

    open_heap = []
    closed = set()

    g_score = {
        INITIAL: 0
    }

    parent = {
        INITIAL: None
    }

    action_taken = {
        INITIAL: None
    }

    first_h = heuristic(INITIAL)

    heappush(
        open_heap,
        (
            first_h,
            first_h,
            next(counter),
            INITIAL
        )
    )

    expansion_number = 0

    while open_heap:

        f_value, h_value, _, current = heappop(open_heap)

        if current in closed:
            continue

        current_g = g_score[current]

        closed.add(current)
        expansion_number += 1

        print(
            f"\nExpansion {expansion_number}: "
            f"{state_string(current)}"
        )

        print(
            f"g = {current_g}, "
            f"h = {heuristic(current)}, "
            f"f = {current_g + heuristic(current)}"
        )

        if is_goal(current):
            print("\nGOAL FOUND")
            return reconstruct_path(
                current,
                parent,
                action_taken,
                g_score
            )

        for action, cost, next_state in successors(current):

            if next_state in closed:
                continue

            tentative_g = current_g + cost

            if tentative_g < g_score.get(
                next_state,
                float("inf")
            ):

                g_score[next_state] = tentative_g
                parent[next_state] = current
                action_taken[next_state] = action

                next_h = heuristic(next_state)
                next_f = tentative_g + next_h

                heappush(
                    open_heap,
                    (
                        next_f,
                        next_h,
                        next(counter),
                        next_state
                    )
                )

                print(
                    f"  Generated: {action:25s} "
                    f"cost={cost}, "
                    f"g={tentative_g}, "
                    f"h={next_h}, "
                    f"f={next_f}"
                )

    return None


def reconstruct_path(
    goal_state,
    parent,
    action_taken,
    g_score
):
    path = []

    current = goal_state

    while current is not None:
        path.append(
            (
                current,
                action_taken[current],
                g_score[current]
            )
        )

        current = parent[current]

    path.reverse()

    return path


def main():
    print("MONKEY AND BANANA - A* SEARCH")
    print("=" * 50)

    print("\nInitial State:")
    print(state_string(INITIAL))

    print("\nStarting A* search...")

    solution = a_star()

    if solution is None:
        print("\nNo solution exists.")
        return

    print("\n" + "=" * 50)
    print("OPTIMAL ACTION SEQUENCE")
    print("=" * 50)

    total_cost = 0

    for index in range(1, len(solution)):
        state, action, cumulative_cost = solution[index]

        previous_cost = solution[index - 1][2]
        action_cost = cumulative_cost - previous_cost

        total_cost += action_cost

        print(
            f"{index}. {action}"
            f" | Cost = {action_cost}"
            f" | State = {state_string(state)}"
        )

    print("\nFinal state:")
    print(state_string(solution[-1][0]))

    print("\nFinal path cost:")
    print(total_cost)

    print("\nOptimality:")
    print(
        "A* found the minimum-cost solution because "
        "the heuristic is admissible and consistent."
    )


if __name__ == "__main__":
    main()

MONKEY AND BANANA - A* SEARCH

Initial State:
(D, W, D, F, -, -)

Starting A* search...

Expansion 1: (D, W, D, F, -, -)
g = 0, h = 10, f = 10
  Generated: Walk D -> C               cost=1, g=1, h=11, f=12
  Generated: Walk D -> W               cost=1, g=1, h=10, f=11
  Generated: Push B2 D -> C            cost=2, g=2, h=8, f=10
  Generated: Push B2 D -> W            cost=2, g=2, h=10, f=12

Expansion 2: (C, W, C, F, -, -)
g = 2, h = 8, f = 10
  Generated: Walk C -> D               cost=1, g=3, h=8, f=11
  Generated: Walk C -> W               cost=1, g=3, h=7, f=10

Expansion 3: (W, W, C, F, -, -)
g = 3, h = 7, f = 10
  Generated: Push B1 W -> D            cost=2, g=5, h=7, f=12
  Generated: Push B1 W -> C            cost=2, g=5, h=5, f=10

Expansion 4: (C, C, C, F, -, -)
g = 5, h = 5, f = 10
  Generated: Walk C -> D               cost=1, g=6, h=5, f=11
  Generated: Walk C -> W               cost=1, g=6, h=5, f=11
  Generated: Push B2 C -> D            cost=2, g=7, h=7, f=14
  Generate